# QFT-Graph: Baseline Comparisons & Generalization Tests

Trains four architectures on the same data and evaluates coupling generalization.

**Purpose:** Generate Tables II and III for the paper.

**Steps:**
1. Setup & load existing MC data
2. Train all four models (HeteroGNN, HomogeneousGNN, LatticeCNN, MLP)
3. Compare energy prediction accuracy
4. Evaluate coupling generalization
5. Save results for paper figures

**Runtime:** ~15–20 min on Colab GPU (T4/V100). All results save directly to Google Drive.

## 0. Setup: Mount Drive & Install Dependencies

In [ ]:
import os
import sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
    !pip install -q torch-geometric omegaconf
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
    print(f'Working directory: {os.getcwd()}')
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)

print('Setup complete.')

In [ ]:
import json
import time
import importlib
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader as PyGDataLoader

from qft_graph.config import (
    LatticeConfig, ScalarFieldConfig, MCConfig, ModelConfig, TrainingConfig
)
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.fields.scalar import ScalarField
from qft_graph.actions.phi4 import Phi4Action
from qft_graph.mc.metropolis import MetropolisSampler, create_sampler
from qft_graph.graphs.builder import HeteroGraphBuilder
from qft_graph.models.hetero_gnn import HeteroGNN
from qft_graph.training.losses import EnergyMatchingLoss
from qft_graph.training.metrics import energy_correlation, relative_error
from qft_graph.utils.reproducibility import set_seed

# Force-reload baseline modules to pick up any code changes on Drive
import qft_graph.models.baselines.homogeneous_gnn as _hgnn_mod
import qft_graph.models.baselines.lattice_cnn as _cnn_mod
import qft_graph.models.baselines.mlp_baseline as _mlp_mod
importlib.reload(_hgnn_mod)
importlib.reload(_cnn_mod)
importlib.reload(_mlp_mod)
from qft_graph.models.baselines.homogeneous_gnn import HomogeneousGNN
from qft_graph.models.baselines.lattice_cnn import LatticeCNN
from qft_graph.models.baselines.mlp_baseline import MLPBaseline

# Verify the skip-connection fix is loaded:
# The action_head's first Linear should take 2*hidden_dim (h0 || h) not hidden_dim
import inspect
src = inspect.getsource(_hgnn_mod.HomogeneousGNN.forward)
assert 'h0 = h' in src and 'h0, h' in src, (
    'ERROR: HomogeneousGNN does not have the skip-connection fix. '
    'Expected h0 = h (save initial encoding) and cat([h0, h]) in forward(). '
    'Check that Drive has synced the latest homogeneous_gnn.py.'
)
print('HomogeneousGNN skip-connection fix verified.')

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

plt.style.use('dark_background')
%matplotlib inline

## 1. Configuration & Data Loading

In [ ]:
# === Parameters (match training notebook) ===
LATTICE_SIZE = (16, 16)
LATTICE_SPACING = 1.0
MASS_SQUARED = -0.5
COUPLING = 0.5
HIDDEN_DIM = 64
N_MP_BLOCKS = 3
ENCODER_LAYERS = 2
EPOCHS = 150
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
N_CONFIGS = 5000

L = LATTICE_SIZE[0]
N_SITES = L * L

print(f'Lattice: {L}x{L} = {N_SITES} sites')
print(f'Physics: m\u00b2={MASS_SQUARED}, \u03bb={COUPLING}')
print(f'Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

In [ ]:
# Load MC data (must already exist from 03_train_colab.ipynb)
dims_str = f'{L}x{L}'
data_path = Path(f'data/mc_configs/phi4_{dims_str}_m2={MASS_SQUARED}_lam={COUPLING}/mc_data.pt')

if not data_path.exists():
    raise FileNotFoundError(
        f'MC data not found at {data_path}. '
        'Run notebook 03_train_colab.ipynb first to generate training data.'
    )

mc_data = torch.load(data_path, weights_only=False)
configurations = mc_data['configurations'][:N_CONFIGS]
actions = mc_data['actions'][:N_CONFIGS]
print(f'Loaded {len(configurations)} configurations from {data_path}')
print(f'Actions: mean={actions.mean():.2f}, std={actions.std():.2f}')

In [ ]:
# Build graph dataset (used by GNN models)
lattice_config = LatticeConfig(dimensions=LATTICE_SIZE, spacing=LATTICE_SPACING)
lattice = HypercubicLattice(lattice_config)
scalar_field = ScalarField()
builder = HeteroGraphBuilder(lattice, [scalar_field])

print('Building graph dataset...')
dataset = builder.build_dataset(
    configurations={'scalar': configurations},
    actions=actions,
)

# 80/20 split
n_train = int(0.8 * len(dataset))
train_dataset = dataset[:n_train]
val_dataset = dataset[n_train:]
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

train_loader = PyGDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = PyGDataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 2. Training Loop (Shared by All Models)

A single training function ensures identical protocols across all architectures.

In [ ]:
def train_and_evaluate(model, model_name, train_loader, val_loader,
                       epochs=EPOCHS, lr=LEARNING_RATE, device=device):
    """Train a model and return metrics + training history."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = EnergyMatchingLoss()

    n_params = sum(p.numel() for p in model.parameters())
    print(f'\n{"="*60}')
    print(f'Training: {model_name} ({n_params:,} parameters)')
    print(f'{"="*60}')

    history = {'train_loss': [], 'val_loss': [], 'val_corr': []}
    start = time.time()

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        total_loss, n_batches = 0.0, 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            output = model(batch)
            loss = criterion(output['energy'], batch.y.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        train_loss = total_loss / max(n_batches, 1)

        # --- Validate ---
        model.eval()
        all_pred, all_true = [], []
        val_loss_sum, val_batches = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                output = model(batch)
                pred = output['energy']
                target = batch.y.to(device)
                val_loss_sum += criterion(pred, target).item()
                val_batches += 1
                all_pred.append(pred.cpu())
                all_true.append(target.cpu())

        preds = torch.cat(all_pred)
        trues = torch.cat(all_true)
        corr = energy_correlation(preds, trues)
        val_loss = val_loss_sum / max(val_batches, 1)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_corr'].append(corr)

        scheduler.step()

        if epoch % 25 == 0 or epoch == 1:
            print(f'  Epoch {epoch:>3d}/{epochs} | '
                  f'Train: {train_loss:.6f} | Val: {val_loss:.6f} | r={corr:.4f}')

    elapsed = time.time() - start
    rel_err = relative_error(preds, trues)

    print(f'  Done in {elapsed:.1f}s. Final r={corr:.4f}, rel_err={rel_err:.4f}')

    result = {
        'model': model_name,
        'n_params': n_params,
        'pearson_r': round(corr, 6),
        'relative_error': round(rel_err, 6),
        'train_time_s': round(elapsed, 1),
    }
    return result, history

## 3. Train All Four Models

In [ ]:
all_results = []
all_histories = {}

### 3a. HeteroGNN (Ours)

In [ ]:
set_seed(42)
hetero_model = HeteroGNN(
    config=ModelConfig(
        hidden_dim=HIDDEN_DIM, n_mp_blocks=N_MP_BLOCKS,
        encoder_layers=ENCODER_LAYERS, activation='gelu', readout='energy',
    ),
    lattice_dim=lattice.dimension(),
    field_types={'scalar': scalar_field.dof_per_site()},
    lattice_spacing=lattice.lattice_spacing(),
)
r, h = train_and_evaluate(hetero_model, 'HeteroGNN (ours)', train_loader, val_loader)
all_results.append(r)
all_histories['HeteroGNN'] = h

### 3b. Homogeneous GNN (Ablation Baseline)

In [ ]:
set_seed(42)
homo_model = HomogeneousGNN(
    lattice_dim=lattice.dimension(),
    hidden_dim=HIDDEN_DIM,
    n_mp_blocks=N_MP_BLOCKS,
    n_encoder_layers=ENCODER_LAYERS,
    lattice_spacing=lattice.lattice_spacing(),
)
r, h = train_and_evaluate(homo_model, 'HomogeneousGNN', train_loader, val_loader)
all_results.append(r)
all_histories['HomogeneousGNN'] = h

### 3c. Lattice CNN

In [ ]:
set_seed(42)
cnn_model = LatticeCNN(
    lattice_dims=LATTICE_SIZE,
    hidden_channels=32,
    n_conv_layers=4,
    lattice_spacing=lattice.lattice_spacing(),
)
r, h = train_and_evaluate(cnn_model, 'LatticeCNN', train_loader, val_loader)
all_results.append(r)
all_histories['LatticeCNN'] = h

### 3d. MLP Baseline

In [ ]:
set_seed(42)
mlp_model = MLPBaseline(
    n_sites=N_SITES,
    hidden_dim=256,
    n_layers=3,
)
r, h = train_and_evaluate(mlp_model, 'MLP', train_loader, val_loader)
all_results.append(r)
all_histories['MLP'] = h

## 4. Comparison Results

This table maps directly to **Table II** in the paper.

In [ ]:
# Print results table
print(f'\n{"="*70}')
print(f'{"ARCHITECTURE COMPARISON RESULTS":^70}')
print(f'{"="*70}')
print(f'{"Model":<22s} {"Params":>10s} {"Pearson r":>12s} {"Rel. Error":>12s} {"Time (s)":>10s}')
print(f'{"-"*70}')
for r in all_results:
    print(f'{r["model"]:<22s} {r["n_params"]:>10,d} {r["pearson_r"]:>12.4f} '
          f'{r["relative_error"]:>12.4f} {r["train_time_s"]:>10.1f}')
print(f'{"="*70}')

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
colors = ['#cc4444', '#4488ff', '#44bb88', '#cc44ff']
names = [r['model'] for r in all_results]
short_names = [n.replace(' (ours)', '') for n in names]

# Bar chart: Pearson r
rs = [r['pearson_r'] for r in all_results]
bars = axes[0].bar(short_names, rs, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
axes[0].set_ylabel('Pearson $r$')
axes[0].set_title('Energy Prediction Accuracy', fontsize=11)
axes[0].set_ylim(min(rs) - 0.01, 1.001)
for bar, val in zip(bars, rs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# Bar chart: Parameter count
params = [r['n_params'] for r in all_results]
axes[1].bar(short_names, [p/1000 for p in params], color=colors, alpha=0.85,
            edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Parameters (thousands)')
axes[1].set_title('Model Size', fontsize=11)

# Training curves
for (name, hist), c in zip(all_histories.items(), colors):
    axes[2].plot(hist['val_corr'], label=name, color=c, alpha=0.85)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Val Pearson $r$')
axes[2].set_title('Training Convergence', fontsize=11)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.2)

for ax in axes[:2]:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.suptitle(f'Baseline Comparison: {L}\u00d7{L} lattice, m\u00b2={MASS_SQUARED}, \u03bb={COUPLING}',
             fontsize=13)
plt.tight_layout()
plt.show()

## 5. Coupling Generalization Test

Evaluate the **trained HeteroGNN** on MC data generated at different $m^2$ values
(without retraining) to test whether it learned a transferable energy functional.

This table maps directly to **Table III** in the paper.

In [ ]:
# Use the model we just trained (or load the colab_run checkpoint)
ckpt_path = Path('experiments/runs/colab_run/model_final.pt')

if ckpt_path.exists():
    print(f'Loading pre-trained model from {ckpt_path}')
    gen_model = HeteroGNN(
        config=ModelConfig(
            hidden_dim=HIDDEN_DIM, n_mp_blocks=N_MP_BLOCKS,
            encoder_layers=ENCODER_LAYERS, activation='gelu', readout='energy',
        ),
        lattice_dim=lattice.dimension(),
        field_types={'scalar': scalar_field.dof_per_site()},
        lattice_spacing=lattice.lattice_spacing(),
    )
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in ckpt:
        gen_model.load_state_dict(ckpt['model_state_dict'])
    else:
        gen_model.load_state_dict(ckpt)
    gen_model = gen_model.to(device)
    print('Loaded saved model.')
else:
    print('No saved model found, using freshly trained HeteroGNN.')
    gen_model = hetero_model  # from section 3a

gen_model.eval()
print('Model ready for generalization evaluation.')

In [ ]:
# Generate or load MC data at each coupling, then evaluate
M2_TEST_VALUES = [-0.3, -0.5, -0.7, -1.0, -1.5, -2.0]
N_GEN_CONFIGS = 2000
data_dir = Path('data/mc_configs')

gen_results = []

for m2 in M2_TEST_VALUES:
    is_train = abs(m2 - MASS_SQUARED) < 1e-6
    dirname = f'phi4_{L}x{L}_m2={m2}_lam={COUPLING}'
    dpath = data_dir / dirname / 'mc_data.pt'

    if dpath.exists():
        print(f'm\u00b2={m2:>5.1f}: Loading existing data...', end=' ')
        md = torch.load(dpath, weights_only=False)
        cfgs = md['configurations'][:N_GEN_CONFIGS]
        acts = md['actions'][:N_GEN_CONFIGS]
    else:
        print(f'm\u00b2={m2:>5.1f}: Generating {N_GEN_CONFIGS} configs...', end=' ')
        lat = HypercubicLattice(LatticeConfig(dimensions=LATTICE_SIZE))
        fc = ScalarFieldConfig(mass_squared=m2, coupling=COUPLING)
        act_fn = Phi4Action(lat, fc)
        mc_cfg = MCConfig(
            n_configs=N_GEN_CONFIGS, n_thermalization=1000,
            n_sweeps_between=10, seed=42,
        )
        samp = create_sampler(act_fn, mc_cfg)
        res = samp.generate(N_GEN_CONFIGS)
        cfgs = res.configurations
        # Compute exact actions
        acts = torch.tensor([act_fn(cfgs[i]) for i in range(len(cfgs))])
        # Save for reuse
        dpath.parent.mkdir(parents=True, exist_ok=True)
        torch.save({'configurations': cfgs, 'actions': acts}, dpath)

    # Build graphs and evaluate
    ds = builder.build_dataset(configurations={'scalar': cfgs}, actions=acts)
    loader = PyGDataLoader(ds, batch_size=64, shuffle=False)

    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            output = gen_model(batch)
            all_pred.append(output['energy'].cpu())
            all_true.append(batch.y.cpu())

    preds = torch.cat(all_pred)
    trues = torch.cat(all_true)
    corr = energy_correlation(preds, trues)
    rel_err = relative_error(preds, trues)

    marker = ' \u2190 training point' if is_train else ''
    print(f'r={corr:.4f}, rel_err={rel_err:.4f}{marker}')

    gen_results.append({
        'm2': m2,
        'lambda': COUPLING,
        'pearson_r': round(corr, 6),
        'relative_error': round(rel_err, 6),
        'n_configs': len(ds),
        'is_training_point': is_train,
    })

In [ ]:
# Print generalization table
print(f'\n{"="*60}')
print(f'{"COUPLING GENERALIZATION RESULTS":^60}')
print(f'{"="*60}')
print(f'{"m\u00b2":<10s} {"Pearson r":>12s} {"Rel. Error":>12s}')
print(f'{"-"*60}')
for r in gen_results:
    marker = ' *' if r['is_training_point'] else ''
    print(f'{r["m2"]:<10.1f} {r["pearson_r"]:>12.4f} {r["relative_error"]:>12.4f}{marker}')
print(f'{"-"*60}')
print('* = training coupling point')

In [ ]:
# Visualize generalization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

m2s = [r['m2'] for r in gen_results]
rs = [r['pearson_r'] for r in gen_results]
errs = [r['relative_error'] for r in gen_results]
is_train = [r['is_training_point'] for r in gen_results]

colors_gen = ['#cc4444' if t else '#4488ff' for t in is_train]
sizes = [120 if t else 60 for t in is_train]

# Left: Pearson r
axes[0].scatter(m2s, rs, c=colors_gen, s=sizes, zorder=5, edgecolors='white', linewidth=0.5)
axes[0].plot(m2s, rs, '--', color='#888888', alpha=0.4, linewidth=1)
axes[0].set_xlabel('$m^2$')
axes[0].set_ylabel('Pearson $r$')
axes[0].set_title('Energy Prediction Accuracy vs Coupling', fontsize=11)
axes[0].set_ylim(min(rs) - 0.05, 1.01)
axes[0].grid(True, alpha=0.2)

# Right: Relative error
axes[1].scatter(m2s, errs, c=colors_gen, s=sizes, zorder=5, edgecolors='white', linewidth=0.5)
axes[1].plot(m2s, errs, '--', color='#888888', alpha=0.4, linewidth=1)
axes[1].set_xlabel('$m^2$')
axes[1].set_ylabel('Relative Error')
axes[1].set_title('Relative Error vs Coupling', fontsize=11)
axes[1].grid(True, alpha=0.2)

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#cc4444',
           markersize=10, label='Training point'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4488ff',
           markersize=8, label='Generalization'),
]
axes[0].legend(handles=legend_elements, fontsize=9)

plt.suptitle(f'Coupling Generalization: trained at m\u00b2={MASS_SQUARED}, \u03bb={COUPLING}',
             fontsize=13)
plt.tight_layout()
plt.show()

## 6. Save All Results

Saves JSON files that `paper/generate_figures.py` can read to produce publication figures.

In [ ]:
output_dir = Path('experiments')
output_dir.mkdir(parents=True, exist_ok=True)

# Baseline comparison results
baseline_path = output_dir / 'baseline_results.json'
with open(baseline_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Baseline results saved to: {baseline_path}')

# Generalization results
gen_path = output_dir / 'generalization_results.json'
with open(gen_path, 'w') as f:
    json.dump(gen_results, f, indent=2)
print(f'Generalization results saved to: {gen_path}')

print(f'\nTo generate paper figures, run from project root:')
print(f'  python paper/generate_figures.py')

## 7. Paper Table Output

Copy-paste ready LaTeX for `paper/main.tex`.

In [ ]:
# Generate LaTeX for Table II (baselines)
print('% === Table II: Architecture Comparison ===')
print('% Copy into paper/main.tex, replacing the placeholder table')
print()
for r in all_results:
    name = r['model']
    if 'ours' in name:
        name = r'\textbf{HeteroGNN (ours)}'
    n = f"{r['n_params']:,}"
    pr = f"{r['pearson_r']:.4f}"
    re_pct = f"{r['relative_error']*100:.2f}\\%"
    print(f'{name} & {n} & {pr} & {re_pct} \\\\')

print()
print('% === Table III: Generalization ===')
print()
for r in gen_results:
    m2_str = f"${r['m2']}$"
    if r['is_training_point']:
        m2_str += r'^\ast'
    pr = f"{r['pearson_r']:.4f}"
    re_pct = f"{r['relative_error']*100:.2f}\\%"
    print(f'{m2_str} & {pr} & {re_pct} \\\\')